# DIAGNOSTIC ERREURS — F1 classe 1 (eligibilite)

**Objectif** : avant de tester une nouvelle technique de rééquilibrage ou une cost function
custom (cf. discussion), comprendre *où* le modèle actuel se trompe. Si les faux négatifs
ressemblent statistiquement aux vrais positifs sur toutes les features disponibles -> plafond
de signal (aucune technique de rééquilibrage/coût ne répare ça). Si un sous-groupe se détache
nettement -> feature manquante ou bug d'encodage à creuser en priorité.

**Pré-requis** : ce notebook est autonome (pas de session Spark) — il relit directement les
caches disque déjà produits par `pipeline_fast_tuning_v1_9.ipynb` :
- `X_val_f32.npy` / `y_val.npy` dans `XY_CACHE_DIR` (cellule 29 du pipeline)
- le modèle retenu (`.joblib`) + ses métadonnées (`_meta.json`) dans `CHECKPOINT_DIR_SKLEARN`
  (cellules 34/37 du pipeline)

**Optionnel mais recommandé** : les vrais noms de features (au lieu de `f_0`, `f_1`, ...).
Voir la cellule tout en bas ("Récupérer les vrais noms de features") — à exécuter UNE FOIS
dans le notebook pipeline principal (celui qui a la session Spark + `df_val` transformé),
ça écrit un `feature_names.json` que ce notebook-ci charge automatiquement s'il existe.


## 1. Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ═══ À adapter si besoin — mêmes valeurs que dans le pipeline principal ═══
XY_CACHE_DIR = "xy_cache"                 # dossier contenant X_val_f32.npy / y_val.npy
CHECKPOINT_DIR_SKLEARN = "checkpoints_sklearn"  # dossier des .joblib + _meta.json

# Nom du modèle à diagnostiquer : "LightGBM" ou "XGBoost" (doit matcher <NOM>.joblib / <NOM>_meta.json)
NOM_MODELE = "LightGBM"

FEATURE_NAMES_PATH = "feature_names.json"  # optionnel, cf. dernière cellule du notebook


## 2. Chargement — X_val/y_val, modèle, seuil

In [ ]:
X_val = np.load(f"{XY_CACHE_DIR}/X_val_f32.npy", mmap_mode="r")
y_val = np.load(f"{XY_CACHE_DIR}/y_val.npy", mmap_mode="r")
X_val = np.asarray(X_val)
y_val = np.asarray(y_val)

modele = joblib.load(f"{CHECKPOINT_DIR_SKLEARN}/{NOM_MODELE}.joblib")
with open(f"{CHECKPOINT_DIR_SKLEARN}/{NOM_MODELE}_meta.json") as f:
    meta = json.load(f)
seuil = meta["seuil"]

print(f"X_val {X_val.shape}, taux positif réel = {y_val.mean():.4%}")
print(f"Modèle : {NOM_MODELE} -- seuil retenu = {seuil:.3f} -- F1 classe 1 (sauvegardé) = {meta['f1_classe1_val']:.4f}")

# Noms de features -- réels si feature_names.json existe (cf. dernière cellule), sinon génériques
try:
    with open(FEATURE_NAMES_PATH) as f:
        feature_names = json.load(f)
    assert len(feature_names) == X_val.shape[1], (
        f"feature_names.json a {len(feature_names)} noms mais X_val a {X_val.shape[1]} colonnes -- "
        f"probablement généré sur une autre version du dataset. Ignoré."
    )
    print(f"{len(feature_names)} noms de features chargés depuis {FEATURE_NAMES_PATH}.")
except (FileNotFoundError, AssertionError) as e:
    feature_names = [f"f_{i}" for i in range(X_val.shape[1])]
    print(f"(pas de noms réels -- {e if isinstance(e, AssertionError) else 'fichier absent'} -- "
          f"noms génériques f_0..f_{X_val.shape[1]-1} utilisés à la place.)")


## 3. Prédictions & répartition FN / FP / TP / TN

In [ ]:
probas = modele.predict_proba(X_val)[:, 1]
preds = (probas >= seuil).astype(int)

idx_tp = np.where((y_val == 1) & (preds == 1))[0]
idx_fn = np.where((y_val == 1) & (preds == 0))[0]   # <- ce qu'on rate (le plus coûteux, classe rare)
idx_fp = np.where((y_val == 0) & (preds == 1))[0]
idx_tn = np.where((y_val == 0) & (preds == 0))[0]

print(f"TP = {len(idx_tp):>7}  (positifs bien détectés)")
print(f"FN = {len(idx_fn):>7}  (positifs ratés -- {len(idx_fn)/(len(idx_fn)+len(idx_tp)):.1%} des positifs)")
print(f"FP = {len(idx_fp):>7}  (négatifs pris pour positifs)")
print(f"TN = {len(idx_tn):>7}  (négatifs bien détectés)")

from sklearn.metrics import classification_report
print()
print(classification_report(y_val, preds, target_names=["0", "1"]))


## 4. Feature importance du modèle

Sert de filtre : on ne compare FN vs TP que sur les features que le modèle utilise
réellement le plus, plutôt que sur les ~200 colonnes en vrac.


In [ ]:
importances = pd.Series(modele.feature_importances_, index=feature_names).sort_values(ascending=False)
top_features = importances.head(25)

plt.figure(figsize=(8, 8))
top_features.iloc[::-1].plot(kind="barh")
plt.title(f"Top 25 features par importance -- {NOM_MODELE}")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

top_features


## 5. FN vs TP -- où sont les positifs qu'on rate ?

C'est la comparaison la plus importante : si les FN ressemblent aux TP sur toutes les
top features (moyennes proches, distributions qui se chevauchent), le modèle ne rate pas
ces clients par manque d'agressivité au seuil -- il ne trouve juste pas de séparation dans
les features actuelles pour ce sous-groupe. Colonne clé : `diff_std` (différence de moyenne
FN-TP, exprimée en écarts-types de la feature) -- au-delà de ~0.3-0.5 en valeur absolue,
la feature discrimine vraiment FN vs TP ; en dessous, elle ne les distingue quasiment pas.


In [ ]:
def comparer_groupes(idx_a, nom_a, idx_b, nom_b, features_a_comparer, top_n=25):
    lignes = []
    for feat in features_a_comparer:
        j = feature_names.index(feat)
        vals_a = X_val[idx_a, j]
        vals_b = X_val[idx_b, j]
        std_pool = np.sqrt((vals_a.var() + vals_b.var()) / 2) + 1e-9
        diff_std = (vals_a.mean() - vals_b.mean()) / std_pool
        lignes.append({
            "feature": feat,
            f"moyenne_{nom_a}": vals_a.mean(),
            f"moyenne_{nom_b}": vals_b.mean(),
            "diff_std": diff_std,
        })
    df_cmp = pd.DataFrame(lignes).sort_values("diff_std", key=np.abs, ascending=False)
    return df_cmp.head(top_n)


cmp_fn_tp = comparer_groupes(idx_fn, "FN", idx_tp, "TP", top_features.index.tolist())
cmp_fn_tp


In [ ]:
# Visuel -- distribution des 6 features qui séparent le plus FN et TP
top6 = cmp_fn_tp.head(6)["feature"].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.ravel(), top6):
    j = feature_names.index(feat)
    ax.hist(X_val[idx_tp, j], bins=40, alpha=0.5, density=True, label="TP (détectés)")
    ax.hist(X_val[idx_fn, j], bins=40, alpha=0.5, density=True, label="FN (ratés)")
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 6. FP vs TN -- sur quoi le modèle "s'emballe" ?

Moins critique que FN (les FP coûtent une relance commerciale inutile, pas une opportunité
manquée), mais utile pour voir si le modèle sur-généralise sur un pattern précis.


In [ ]:
cmp_fp_tn = comparer_groupes(idx_fp, "FP", idx_tn, "TN", top_features.index.tolist())
cmp_fp_tn


## 7. Calibration -- le modèle est-il au moins bien rangé (même s'il n'est pas parfait) ?

Si le PR-AUC est correct mais le F1 au seuil optimal reste bas, ça confirme un problème de
séparabilité (chevauchement de classes) plutôt qu'un problème de seuil/coût. Complète le
diagnostic ci-dessus.


In [ ]:
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score

precisions, recalls, seuils_pr = precision_recall_curve(y_val, probas)
pr_auc = auc(recalls, precisions)
roc_auc = roc_auc_score(y_val, probas)
taux_base = y_val.mean()

print(f"PR-AUC  = {pr_auc:.4f}  (référence -- modèle aléatoire = taux de base = {taux_base:.4f})")
print(f"ROC-AUC = {roc_auc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(recalls, precisions)
axes[0].axhline(taux_base, color="grey", linestyle="--", label=f"baseline={taux_base:.3f}")
axes[0].set_xlabel("rappel"); axes[0].set_ylabel("précision")
axes[0].set_title(f"Courbe précision-rappel (AUC={pr_auc:.3f})")
axes[0].legend()

axes[1].hist(probas[y_val == 0], bins=50, alpha=0.5, density=True, label="classe 0")
axes[1].hist(probas[y_val == 1], bins=50, alpha=0.5, density=True, label="classe 1")
axes[1].axvline(seuil, color="black", linestyle="--", label=f"seuil retenu={seuil:.3f}")
axes[1].set_xlabel("probabilité prédite")
axes[1].set_title("Séparation des probabilités par vraie classe")
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Lecture des résultats -- quoi faire selon ce que tu vois

- **`diff_std` faible partout (< ~0.3) dans les sections 5 et 6** : les FN ne se distinguent
  pas des TP sur les features actuelles -> plafond de signal. Focal loss / SMOTE / stacking
  n'y changeront pas grand-chose. Piste : chercher des features supplémentaires dans les 21
  tables sources, pas une nouvelle technique de rééquilibrage.
- **1-2 features avec `diff_std` élevé et cohérent** (ex. une tranche d'âge, une ancienneté)
  qui ressort dans les FN : regarde si cette feature a des trous/valeurs aberrantes pour ce
  sous-groupe (bug d'encodage, imputation) avant de conclure à un manque de signal réel.
- **Nuage de points bimodal net dans l'histogramme des probabilités (section 7)** : bon signe,
  le modèle sépare -- le F1 bas vient alors surtout du déséquilibre extrême (~4.5%), pas d'un
  manque de séparabilité ; là, retravailler seuil/coût a plus de chances de payer.
- **Chevauchement quasi total** : le plafond n'est pas un problème de seuil ni de coût.


---
## Annexe -- récupérer les vrais noms de features (optionnel)

À coller et exécuter **dans le notebook pipeline principal** (celui avec la session Spark),
juste après la construction du pipeline d'encodage (section 6, après la cellule qui définit
`assembler`/`feature_cols`) -- PAS dans ce notebook-ci, qui n'a pas de session Spark.

Spark n'expose pas les noms de colonnes expansées par `OneHotEncoder` (ex. `pack_actuel_ohe`
devient plusieurs colonnes binaires) directement dans `feature_cols` -- il faut les lire dans
les métadonnées attachées à la colonne vectorielle `features` après un `.transform()`.


In [ ]:
# ═══ À exécuter dans le notebook PIPELINE PRINCIPAL (avec Spark) -- pas ici ═══
from pyspark.ml.attribute import AttributeGroup
import json

# n'importe quel DataFrame déjà passé par l'encodeur fonctionne (ex. df_fit après la
# construction du pipeline section 6, ou l'output de encodeur_commun.transform(df_val))
df_pour_metadata = encodeur_commun.transform(df_val.limit(1))  # ou df_fit, peu importe

attr_group = AttributeGroup.fromStructField(df_pour_metadata.schema["features"])
feature_names_reels = [a.name for a in attr_group.attributes]

print(f"{len(feature_names_reels)} noms extraits (doit matcher X_val.shape[1] du notebook diagnostic).")

with open("feature_names.json", "w") as f:
    json.dump(feature_names_reels, f, indent=2)
print("Écrit -> feature_names.json (à copier a côté du notebook diagnostic, même dossier que XY_CACHE_DIR).")
